In [ ]:
!sudo apt update
!sudo apt install python3-pip libopencv-dev
!pip install ultralytics opencv-python

In [ ]:
from ultralytics import YOLO
from picamera2 import Picamera2
import cv2
import numpy as np
import time
from IPython.display import clear_output, display, Image

# 1️⃣ YOLO 모델 로드
model = YOLO("best.pt")  # COCO 사전학습 모델...였는데 그냥 파인 튜닝 모델로

# 2️⃣ PiCamera2 설정
picam2 = Picamera2()
video_config = picam2.create_video_configuration(main={"format": "RGB888", "size": (640, 480)})
picam2.configure(video_config)
picam2.start()
time.sleep(1)

try:
    while True:
        frame = picam2.capture_array()
        frame = cv2.rotate(frame, cv2.ROTATE_180)  # 필요 시 제거
        annotated = frame.copy()

        results = model.predict(frame, imgsz=640, conf=0.4)[0]

        for box in results.boxes:
            cls_id = int(box.cls[0])
            class_name = results.names[cls_id]

            if class_name != "traffic light":
                continue

            x1, y1, x2, y2 = map(int, box.xyxy[0])
            roi = frame[y1:y2, x1:x2]

            # HSV 변환 후 가로 방향 3등분
            hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
            h, w = hsv.shape[:2]

            left  = hsv[:, 0:w//3]
            mid   = hsv[:, w//3:2*w//3]
            right = hsv[:, 2*w//3:]

            v_left  = np.mean(left[:, :, 2])
            v_mid   = np.mean(mid[:, :, 2])
            v_right = np.mean(right[:, :, 2])

            v_values = [v_left, v_mid, v_right]
            max_idx = np.argmax(v_values)

            if max(v_values) < 60:
                color = "UNKNOWN"
            else:
                # 가로 방향 기준: 좌 → 우 = GREEN, YELLOW, RED
                color = ["GREEN", "YELLOW", "RED"][max_idx]

            print(f"Detected Traffic Light Color: {color}")
            label = f"{class_name} ({color})"
            cv2.rectangle(annotated, (x1, y1), (x2, y2), (0, 255, 255), 2)
            cv2.putText(annotated, label, (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
           # break  # 첫 번째 신호등만 인식

        # 표시된 이미지를 가로로 출력
        _, jpeg = cv2.imencode('.jpg', annotated)
        clear_output(wait=True)
        display(Image(data=jpeg.tobytes()))
        time.sleep(0.1)

except KeyboardInterrupt:
    print("중단됨.")
finally:
    picam2.stop()
    picam2.close()